In [1]:
!pip install langchain_huggingface

In [2]:
# Importing the Module
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from langchain_core.prompts import PromptTemplate

In [3]:
# Importing the Model
llm = HuggingFacePipeline.from_model_id(
    model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task = "text-generation"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [4]:
model = ChatHuggingFace(llm=llm)

In [5]:
output = model.invoke("What is the capital of Pakistan?")
output.content

'<|user|>\nWhat is the capital of Pakistan?</s>\n<|assistant|>\nThe capital of Pakistan is Islamabad.'

In [ ]:
# Prompt template
prompt = """
Please summarize the research paper titled "{paper_input}" with the following specifications:
Explanation Style: {style_input}
Explanation Length: {length_input}
1. Mathematical Details:
    - Include relevant mathematical equations if present in the paper.
    - Explain the mathematical concepts using simple, intuitive code snippets where applicable.
2. Analogies:
    - Use relatable analogies to simplify complex ideas.
If certain information is not available in the paper, respond with: "Insufficient information available" instead of guessing.
Ensure the summary is clear, accurate, and aligned with the provided style and length.
"""

In [6]:
# Creating a chat model
# The problem here is that we are not maintaining a chat history for future conversation
# As we can see it forgot the previous chat
while True:
  user_input = input("You: ")
  if user_input == 'exit':
    break

  result = model.invoke(user_input)
  print("AI: ",result.content)

You: What is difference between framework and library
AI:  <|user|>
What is difference between framework and library</s>
<|assistant|>
A framework and a library are two different types of software development components that are designed to provide a common set of tools and techniques for building complex applications. Here are some key differences between these two types of software components:

1. Functionality: Frameworks and libraries are designed to provide a specific set of features and functionalities that are pre-built, pre-configured, and pre-tested. They usually include a set of ready-to-use classes, methods, and functions that are used to build complex applications. In contrast, frameworks and libraries are designed to be customizable, allowing developers to choose which features and functionalities they want to incorporate into their applications.

2. Development Process: Both frameworks and libraries can be developed using different development methodologies, but they are 

In [8]:
# A simple solution can be to save the chat in list

chat_history = []

while True:
  user_input = input("You: ")
  chat_history.append(user_input)

  if user_input == 'exit':
    break

  result = model.invoke(chat_history)
  chat_history.append(result.content)
  print("AI: ",result.content)

print(chat_history)

You: What is difference between framework and library
AI:  <|user|>
What is difference between framework and library</s>
<|assistant|>
In software development, the term framework and library refers to two different types of software components. Frameworks are pre-built software components that are designed to provide a set of services or features to other applications. These services can be accessed by other applications without the need for custom development. The main purpose of framework is to provide a uniform and standardized way of accessing a set of services.

On the other hand, libraries are software components that provide a set of pre-defined functions or methods that can be used by other applications. The main purpose of a library is to provide a standardized and flexible way of accessing a set of functions or methods. The most common examples of libraries are the C++ standard library and the Java Collections Framework.

In general, frameworks are designed to provide a set o

In [11]:
# The above code is keeping the chat history but the problem is it is not telling us what message is by human which is Ai message etc.
# To Solve this problem, There are multiple class in langchain
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

message = [
    SystemMessage(content='You are an helpful assistent'),
    HumanMessage(content="Tell me about Python")
]

result = model.invoke(message)
message.append(AIMessage(content=result.content))

print(result.content)

<|system|>
You are an helpful assistent</s>
<|user|>
Tell me about Python</s>
<|assistant|>
Python is a programming language that is widely used for web development, data science, and artificial intelligence. It is known for its simplicity, readability, and extensibility. Python is an interpreted language, meaning that it runs directly on the computer rather than being compiled to machine code. This makes it faster and more efficient than other compiled languages. It has a large and vibrant community, which has developed a vast library of tools, frameworks, and libraries that make it easy to build complex applications. Some of the key features of Python include:

1. Strong focus on data structures and algorithms: Python's emphasis on data structures and algorithms has made it a popular choice for data science, machine learning, and artificial intelligence applications.

2. Simple syntax: Python's syntax is simple and easy to learn. This makes it a popular choice for web development, wh

In [12]:
message

[SystemMessage(content='You are an helpful assistent', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Tell me about Python', additional_kwargs={}, response_metadata={}),
 AIMessage(content="<|system|>\nYou are an helpful assistent</s>\n<|user|>\nTell me about Python</s>\n<|assistant|>\nPython is a programming language that is widely used for web development, data science, and artificial intelligence. It is known for its simplicity, readability, and extensibility. Python is an interpreted language, meaning that it runs directly on the computer rather than being compiled to machine code. This makes it faster and more efficient than other compiled languages. It has a large and vibrant community, which has developed a vast library of tools, frameworks, and libraries that make it easy to build complex applications. Some of the key features of Python include:\n\n1. Strong focus on data structures and algorithms: Python's emphasis on data structures and algorithms has mad

In [16]:
# Let's integrate message in our ChatBot
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

chat_history = [
    SystemMessage(content="You're a helpful assistent")
]

while True:
  user_input = content=input("You: ")
  chat_history.append(HumanMessage(content=user_input))
  if user_input == 'exit':
    break

  result = model.invoke(chat_history)
  chat_history.append(AIMessage(content=result.content))
  print("AI: ",result.content)

print(chat_history)

You: What is 10 + 20
AI:  <|system|>
You're a helpful assistent</s>
<|user|>
What is 10 + 20</s>
<|assistant|>
10 + 20 = 30
You: What is the capital of USA
AI:  <|system|>
You're a helpful assistent</s>
<|user|>
What is 10 + 20</s>
<|assistant|>
<|system|>
You're a helpful assistent</s>
<|user|>
What is 10 + 20</s>
<|assistant|>
10 + 20 = 30</s>
<|user|>
What is the capital of USA</s>
<|assistant|>
The capital of USA is Washington, D.C.
You: exit


[SystemMessage(content="You're a helpful assistent", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is 10 + 20', additional_kwargs={}, response_metadata={}),
 AIMessage(content="<|system|>\nYou're a helpful assistent</s>\n<|user|>\nWhat is 10 + 20</s>\n<|assistant|>\n10 + 20 = 30", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What is the capital of USA', additional_kwargs={}, response_metadata={}),
 AIMessage(content="<|system|>\nYou're a helpful assistent</s>\n<|user|>\nWhat is 10 + 20</s>\n<|assistant|>\n<|system|>\nYou're a helpful assistent</s>\n<|user|>\nWhat is 10 + 20</s>\n<|assistant|>\n10 + 20 = 30</s>\n<|user|>\nWhat is the capital of USA</s>\n<|assistant|>\nThe capital of USA is Washington, D.C.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='exit', additional_kwargs={}, response_metadata={})]

In [20]:
# Chat Prompt Template: It is class inside prompt. It is used list of message be dynamic.
from langchain_core.prompts import ChatPromptTemplate

chat_history = ChatPromptTemplate([
    ("system","You are a helpful {domain} expert"),
    ("human","Explain in simple language, what is {topic}")
])

prompt = chat_history.invoke({"domain":"Teacher","topic":"Verb"})

print(prompt)

messages=[SystemMessage(content='You are a helpful Teacher expert', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain in simple language, what is Verb', additional_kwargs={}, response_metadata={})]


In [24]:
# Message PlaceHolder: It is a class which load chat history from database
from langchain_core.prompts import MessagesPlaceholder, ChatPromptTemplate
from langchain_core.messages import HumanMessage

# Creating Chat template
chat_template = ChatPromptTemplate([
    ('system',"You are a helpful customer support agent"),
    MessagesPlaceholder(variable_name="chat_history"),
    ('user', "{query}")
])

# Load chat history
chat_history = []
with open('chat_history.txt','r') as f:
  chat_history.extend(f.readlines())

# create prompt
chat_template.invoke({"chat_history":chat_history, "query":"Where is my refund."})

ChatPromptValue(messages=[SystemMessage(content='You are a helpful customer support agent', additional_kwargs={}, response_metadata={}), HumanMessage(content='HumanMessage(content="I want to request a refund for my order #12345.")\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='AIMessage(content="Your refund request for order #12345 has been initiated. It will be processed in 3-5 business days.")', additional_kwargs={}, response_metadata={}), HumanMessage(content='Where is my refund.', additional_kwargs={}, response_metadata={})])